<a href="https://colab.research.google.com/github/rakshitha2006gowda-art/Banking-FAQ-s-Assistant1/blob/main/Banking_FAQ's_Assistant_File2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Policy-Bound Decision System for Banking Customer Support**

The AI must:

Answer only when context is sufficient

Refuse when critical context is missing

Ask clarification questions instead of guessing

Never invent banking policies, balances, transactions, or eligibility

### **Why This Task Matters**

## In real banking systems:

Customer information may be incomplete

Transaction details may be missing

User information may conflict

Banking policies must be followed exactly

AI must NOT guess or hallucinate

This task focuses on controlled failure, not just better answers.

# **Step 1: Add Guardrails to Static Context**

We enforce when the AI should refuse to answer.

In [6]:
SYSTEM_CONTEXT_GUARDED = """
You are a banking customer support assistant.

Rules:
- Do not assume missing information
- If required data is missing, ask a clarification question
- If policy cannot be applied, respond with "Unable to determine"
- Never hallucinate transaction details, balances, fees, or eligibility
- Do not make financial decisions without sufficient context
- Follow the provided banking policy only
- Be polite and professional
"""

### **Step 2: Incomplete User Context**
Here, we intentionally remove critical information.

In [7]:
user_query = "Why was I charged a fee?"

user_profile_incomplete = {
    "role": "bank customer"
    # Missing transaction type
    # Missing transaction date
    # Missing fee amount
}

### **Step 3: Add Banking Policy**

In [8]:
BANKING_POLICY = """
Banking Fee Policy:

- ATM withdrawal fees may apply when using certain non-bank ATMs
- International transactions may include foreign transaction fees
- Some account types may have monthly maintenance fees
- Fee eligibility depends on the transaction type and account type
- The exact fee cannot be determined without sufficient transaction information
"""

### **Step 4: Assemble Context with Missing Information**

Notice that we DO NOT fill in the missing values.

In [9]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{BANKING_POLICY}

Customer Profile:
- Role: {user_profile_incomplete['role']}

Customer Question:
{user_query}
"""

In [11]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# 3. Send request to the model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ]
)

# 4. Print response
print(response.choices[0].message.content)

To help me understand why you were charged a fee, could you please provide some more details about the transaction?

Specifically, could you tell me:
1.  What type of transaction was it (e.g., ATM withdrawal, online purchase, monthly fee)?
2.  Approximately when did the fee occur (date or time)?
3.  Which account was involved?

Once I have this information, I can look into it for you.


## **What Should Happen?**

# Correct behavior:

AI asks for missing information

AI says "Unable to determine"

AI does not invent transaction information

**Controlled refusal is a success.**

## **Step 6: Conflicting Context**

Now we simulate contradictory information.

In [12]:
user_profile_conflict = {
    "role": "bank customer",
    "account_type": "Basic Savings Account",
    "transaction_type": "ATM withdrawal",
    "atm_type": "Bank ATM",
    "fee_amount": "₹25",
    "transaction_days_ago": 2
}

In [13]:
ATM_POLICY = """
ATM Fee Policy:

- ATM withdrawals from the bank's own ATMs are free
  for Basic Savings Accounts.
- Fees may apply when using other-bank ATMs.
- If transaction records conflict with the policy,
  do not guess the reason for the charge.
- The transaction should be verified before giving a final explanation.
"""

final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{ATM_POLICY}

Customer Profile:
- Role: {user_profile_conflict['role']}
- Account Type: {user_profile_conflict['account_type']}
- Transaction Type: {user_profile_conflict['transaction_type']}
- ATM Type: {user_profile_conflict['atm_type']}
- Fee Charged: {user_profile_conflict['fee_amount']}
- Transaction Date: {user_profile_conflict['transaction_days_ago']} days ago

Customer Question:
Why was I charged an ATM fee?
"""

In [14]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_conflict
        }
    ]
)

print(response.choices[0].message.content)

Thank you for reaching out.

According to our policy, ATM withdrawals from our bank's own ATMs are free for Basic Savings Accounts. However, your transaction details indicate a fee of ₹25 was charged for an ATM withdrawal from a Bank ATM.

Since this conflicts with our standard policy, I'm unable to provide a definitive reason for the charge at this moment. We will need to verify the transaction details to understand why the fee was applied.

Would you like me to initiate a review of this specific transaction for you?
